In [ ]:
import os

import json
import math
import pickle
import networkx as nx
import pandas as pd
import numpy as np

import seaborn as sns
import networkx as nx
from sklearn.preprocessing import StandardScaler

from data.reference.residue_dictionary import residue3to1 as AA3TO1
from data.reference.residue_dictionary import residue1to3 as AA1TO3

import matplotlib.pyplot as plt
from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_sample, get_uniprot_from_nodes, get_pos_from_nodes, get_res_from_nodes, get_node_id_rm_copy

from vizutils import *

from pyaaindex.api import get_features, to_frame
from helper import get_aainfo

In [ ]:
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

aa_seq = np.array(['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y'])

In [ ]:
# Graph Load
finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph.pkl"))
node_in_G = list(finalG.nodes())
node_df_inG = pd.DataFrame(node_in_G, columns=['node_id'])

# origin Data
mut_df = pd.read_csv((f"{DATABASE}/node_mutation_occurrence_essential.csv"))
y_df = pd.read_csv(f"{DATABASE}/node_features_with_location_nodeid_v031026.csv")
edge_df = nx.to_pandas_edgelist(finalG)

# Processed Data
ss_df = pd.read_csv(f"{DATABASE}/processed_ss_df_v040126.csv")
gnomad_mut_df = pd.read_csv((f"{DATABASE}/gnomad_mutation_counts_freq_final.csv"))

# AlphaMissense
with open(f"{DATABASE}/am_features.json", 'rb') as f:
    am = json.load(f)

# Graph EDA

In [ ]:
cc_list = list(nx.connected_components(finalG))
len(cc_list)

In [ ]:
n_in_cc = []
for cc_ in cc_list:
    n_in_cc.append(len(cc_))
n_in_cc.sort(reverse=True)

print("Top5 Size", n_in_cc[:5])

In [ ]:
plt.figure(figsize=(7,3))
plt.hist(n_in_cc[5:], color='skyblue', edgecolor='black', bins=100)
plt.yscale('log')
plt.show()

# Filtering Non-reviewed Protein

In [ ]:
rm_uniprot = [
    "a0a1x8xl64", # AM
    "b4dr52", # AM
    "b2r5b3", # AM
    "q9haw4", # SS
]

In [ ]:
# proc_edge_df = pd.read_csv(f"{DATABASE}/step5_rmStrangeEdge_exceptCDSFilter_human_aa_edges_exclubq_Nucleosome_related_data_v031626.csv")
# proc_edge_df.head(3)
# len(proc_edge_df)

In [ ]:
# rm_nonReview_df = proc_edge_df[
#     (~proc_edge_df['remove_homo_uniprot1'].isin(rm_uniprot)) & 
#     (~proc_edge_df['remove_homo_uniprot2'].isin(rm_uniprot))
# ]
# len(rm_nonReview_df)

In [ ]:
# from generation.graph.helper import GenerateGraph_from_edge
# newG = GenerateGraph_from_edge(rm_nonReview_df, src='nodeid_1', tar='nodeid_2', weight_col='cleaned_total_energy')
# print(newG)

In [ ]:
# with open(os.path.join(DATABASE, 'cleaned_weighted_graph.pkl'), 'wb') as f:
#     pickle.dump(newG, f)

In [ ]:
# g = load_graph(os.path.join(DATABASE, 'cleaned_weighted_graph.pkl'))

In [ ]:
# print(g)

# Mutation Profile

## Overall Analysis

## AAindex

### AAindex1

### AAindex2

In [ ]:
ids = [
    'CSEM940101', # Replaceability
    'GEOD900101', # Hydrophobicity Scoring Matrix
    'GRAR740104', # Chemical Distance
    'JOHM930101', # Strcture-based amino acid scoring Table
    'MIYS930101', # Base-substitution-protein-stability matrix
    'QU_C930103', # Mutant distance based on spatial preference factor
    'GONG920101', # Mutation Matrix for initailly aligning
    # 'KLEP840101' # Net Chard --> First Index (AAindex1)
]
aaidx2 = get_features(ids)

In [ ]:
frames = to_frame(aaidx2['idx2'])
frame_list = []
for feature_id, df in frames.items():
    frame_list.append(df)

In [ ]:
frames['CSEM940101']

In [ ]:
feature_vectors = {}
for matrix_id in ids:
    feature_vectors[matrix_id] = get_upper_triangle_values(frames[matrix_id])

df_features = pd.DataFrame(feature_vectors)
corr_result = df_features.corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_result, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Similarity between Mutation Profiles (Feature-wise)")
plt.show()

## Alpha Missense

In [ ]:
scaler = StandardScaler()

feat_for_sim_1 = []
feat_for_sim_2 = []

# AlphaMissense
mask = (aa_seq != res_1) & (aa_seq != res_2)

am_1 = np.array(am[get_node_id_rm_copy(target_1)])[mask]
am_2 = np.array(am[get_node_id_rm_copy(target_2)])[mask]

# Merge & Normalization 
feat_for_sim_1.append(am_1)
feat_for_sim_2.append(am_2)

feat_for_sim_1 = (feat_for_sim_1 - np.mean(feat_for_sim_1, axis=1, keepdims=True)) / (np.std(feat_for_sim_1, axis=1, keepdims=True) + 1e-8)
feat_for_sim_2 = (feat_for_sim_2 - np.mean(feat_for_sim_2, axis=1, keepdims=True)) / (np.std(feat_for_sim_2, axis=1, keepdims=True) + 1e-8)

In [ ]:
print(feat_for_sim_1)
print(feat_for_sim_2)

In [ ]:
target_1, target_2, RVcoefficient(feat_for_sim_1, feat_for_sim_2)

## Gnomad Mutation

### Manual ADD Mutation Information

In [ ]:
# from generation.mutation.gnomad.helper import process_gnomad_csv
# add_files = ['O60814_H2BC12_gnomAD_v4.1.1_ENST00000356950.csv',
#              'P11388_TOP2A_gnomAD_v4.1.1_ENST00000423485.csv',
#              'Q5EE01_CENPW_gnomAD_v4.1.1_ENST00000368328.csv',
#              'Q96KQ7_EHMT2_gnomAD_v4.1.1_ENST00000375537.csv'
#              ]

In [ ]:
# c_gnomad_mut_df = gnomad_mut_df.copy()
# for file in add_files:
#     print(file)
#     uniprot = file.split("_")[0]
#     ENST = file.split("_")[-1].split(".")[0]
#     gene_symbol= file.split("_")[1]
#     df_raw = pd.read_csv(f"reference/{file}")
#     proc_df = process_gnomad_csv(df_raw, uniprot, ENST, gene_symbol=gene_symbol)
#     c_gnomad_mut_df = pd.concat([c_gnomad_mut_df, proc_df], axis=0)
# c_gnomad_mut_df.to_csv('proc_data/gnomad_mutation_counts_freq_final.csv', index=False)
# c_gnomad_mut_df.tail(5)

### EDA

In [ ]:
gnomad_mut_df.head(4)

In [ ]:
covered_nodes = []
node_in_gnomad = gnomad_mut_df['node_id'].tolist()

for n in node_in_G:
    cleaned_node_id = get_node_id_rm_copy(n)
    if cleaned_node_id in node_in_gnomad:
        covered_nodes.append(n)

In [ ]:
print(len(covered_nodes), "||", f"{round((len(covered_nodes) / len(node_in_G))*100, 2)}%")

In [ ]:
len(gnomad_mut_df.uniprot_id.unique())

# Secondary Strcture

In [ ]:
float_cols = ['node_id', 
              'rel_sasa', 'ss_helix', 'ss_sheet','ss_loop', 'depth', 'hse_up', 'hse_down',
              'dssp_accessibility', 'dssp_TCO', 'dssp_kappa','dssp_alpha', 'dssp_phi', 'dssp_psi',]

str_cols = ['node_id', 
            'dssp_sec_struct', # Class
            'dssp_helix_3_10', 'dssp_helix_alpha', 'dssp_helix_pi', 'dssp_helix_pp', 'dssp_bend',
            'dssp_chirality', 'dssp_sheet', 'dssp_strand', 'dssp_ladder_1','dssp_ladder_2',]

all_cols = list(set(float_cols).union(str_cols))
all_cols.remove('node_id')

In [ ]:
ss_apply_G = pd.merge(node_df_inG, ss_df, on='node_id', how='left')
ss_apply_G

In [ ]:
for col in float_cols:
    print(col, round((len(ss_apply_G[~ss_apply_G[col].isna()])/len(ss_apply_G))*100,2),'%')

print("-----------------------------")
for col in str_cols:
    print(col, round((len(ss_apply_G[~ss_apply_G[col].isna()])/len(ss_apply_G))*100,2),'%')

In [ ]:
ss_nan_nodes = ss_apply_G[ss_apply_G[all_cols].isna().all(axis=1)].node_id.tolist()
nan_uniprot = set([get_uniprot_from_nodes(n) for n in ss_nan_nodes])
print("NaN Node:", len(ss_nan_nodes), nan_uniprot)

In [ ]:
with open('reference/ss_nan_nodes.json', 'w') as f:
    json.dump(ss_nan_nodes, f)

## float EDA

In [ ]:
float_ss_df = ss_apply_G[float_cols]
float_ss_df.info()

In [ ]:
float_ss_df_inG = float_ss_df[float_ss_df['node_id'].isin(node_in_G)]
print(len(node_in_G))
float_ss_df_inG.info()

In [ ]:
plot_distribution_subplots(float_ss_df_inG, set(float_cols).difference('node_id'), cols_per_row=3, log_scale=True)

## str EDA

In [ ]:
str_ss_df = ss_apply_G[str_cols]
str_ss_df.info()

In [ ]:
plot_value_counts(str_ss_df, set(str_cols).difference('node_id'), cols_per_row=2, top_n=15)

# Evolutionary Information

In [ ]:
with open('proc_data/pssm_features.json') as f:
    #  Information Content(Entropy, highscore-preservative) 1 + A, R, N, D, C, Q, E, G, H, I, L, K, M, F, P, S, T, W, Y, V (log-odd score 20)==> len 21
    pssm = json.load(f)

with open('proc_data/hmm_features.json') as f:
    # A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y
    hmm = json.load(f)

## PSSM EDA

In [ ]:
node_in_pssm = list(pssm.keys())
node_in_pssm[0]

In [ ]:
pssm['q14676_2032_ser'
]

In [ ]:
pssm_node_applyG = set(node_in_G).difference(node_in_pssm)
len(pssm_node_applyG)

In [ ]:
for nan_node in pssm_node_applyG:
    original_key = get_node_id_rm_copy(nan_node)
    org_pssm = pssm.get(original_key)
    
    if org_pssm is not None:
        pssm[nan_node] = list(org_pssm) 
    else:
        continue

In [ ]:
node_in_pssm = list(pssm.keys())
pssm_node_applyG = list(set(node_in_G).difference(node_in_pssm))
print(len(pssm_node_applyG))
pssm_node_applyG[:10]

In [ ]:
nan_uniprot_pssm = list(set([get_uniprot_from_nodes(n) for n in pssm_node_applyG]))
print(len(nan_uniprot_pssm))
nan_uniprot_pssm[:10]

In [ ]:
fasta_file_list = os.listdir(FASTA_PATH)
except_fasta = []

for _pssm in nan_uniprot_pssm:
    if f"{_pssm.upper()}.hhr" not in fasta_file_list:
        except_fasta.append(_pssm)


In [ ]:
len(except_fasta)

In [ ]:
except_fasta

## HMM EDA

In [ ]:
node_in_hmm = list(hmm.keys())
len(node_in_hmm), len(node_in_G)

In [ ]:
node_in_hmm = [n.lower() for n in node_in_hmm]

In [ ]:
len(set(node_in_G).difference(set(node_in_hmm)))

In [ ]:
node_in_G_rmCopy = [get_node_id_rm_copy(n) for n in node_in_G]
len(set(node_in_G_rmCopy).difference(set(node_in_hmm)))

In [ ]:
temp = set(node_in_G_rmCopy).difference(set(node_in_hmm))

In [ ]:
temp = set([get_uniprot_from_nodes(n) for n in temp])

In [ ]:
temp